# 🏉 XVPRO — Entraînement YOLOv8 Custom

Ce notebook entraîne un modèle YOLOv8 pour détecter :
- `team_a` — joueurs équipe A
- `team_b` — joueurs équipe B
- `ball` — ballon de rugby
- `referee` — arbitre

**Prérequis :** Avoir annoté ~200 frames sur Roboflow et avoir la clé API Roboflow.

> ⚠️ Assure-toi d'activer le GPU : `Runtime > Change runtime type > T4 GPU`

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer les dépendances

In [ ]:
!pip install ultralytics==8.3.0 roboflow supervision -q

## 3. Importer le dataset Roboflow

👉 **Remplace `YOUR_API_KEY` par ta clé API Roboflow** (Settings → Roboflow API → Copy API Key)

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY"  # ← remplace ici
WORKSPACE = "valentins-workspace-a8u2y"  # ton workspace Roboflow
PROJECT = "xvpro-rugby-ev1en"           # ton projet
VERSION = 1                              # version du dataset

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(f"Dataset téléchargé : {dataset.location}")

## 4. Vérifier le dataset

In [ ]:
import os

dataset_path = dataset.location
print(f"Train : {len(os.listdir(os.path.join(dataset_path, 'train/images')))} images")
print(f"Valid : {len(os.listdir(os.path.join(dataset_path, 'valid/images')))} images")

# Afficher le fichier de config
with open(os.path.join(dataset_path, 'data.yaml')) as f:
    print(f.read())

## 5. Entraîner YOLOv8

In [ ]:
from ultralytics import YOLO

# Charge YOLOv8n (nano) — rapide à entraîner sur Colab
model = YOLO("yolov8n.pt")

results = model.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=100,          # 100 epochs suffisent pour un bon modèle
    imgsz=640,           # taille standard YOLOv8
    batch=16,            # batch size adapté au GPU T4
    name="xvpro_rugby",  # nom de l'expérience
    patience=20,         # stop si pas d'amélioration après 20 epochs
    save=True,
    device=0,            # GPU
    workers=2,
    verbose=True
)

print("✅ Entraînement terminé !")

## 6. Évaluer le modèle

In [ ]:
# Métriques sur le set de validation
metrics = model.val()
print(f"mAP50 : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")

## 7. Tester sur une frame

In [ ]:
import glob
from IPython.display import Image, display

# Prend une image de validation pour tester
test_images = glob.glob(os.path.join(dataset.location, "valid/images/*.jpg"))[:3]

for img_path in test_images:
    result = model.predict(img_path, conf=0.4, save=True, project="/content/predictions")
    print(f"Détections : {len(result[0].boxes)} objets")

# Affiche la dernière prédiction
pred_imgs = glob.glob("/content/predictions/**/*.jpg", recursive=True)
if pred_imgs:
    display(Image(pred_imgs[-1]))

## 8. Trouver et télécharger le modèle

Le meilleur modèle est sauvegardé dans `runs/detect/xvpro_rugby/weights/best.pt`

In [ ]:
import glob

# Trouve le meilleur modèle
best_model = glob.glob("/content/runs/detect/xvpro_rugby*/weights/best.pt")
if best_model:
    print(f"✅ Modèle trouvé : {best_model[0]}")
else:
    print("❌ Modèle non trouvé")

In [ ]:
# Télécharge le modèle sur ton PC
from google.colab import files

if best_model:
    files.download(best_model[0])
    print("📥 Téléchargement lancé — sauvegarde le fichier best.pt")

## 9. Déployer sur Railway

Une fois `best.pt` téléchargé :

1. Place `best.pt` dans le dossier `worker/` de ton projet XVPRO
2. Dans `worker/main.py`, change :
   ```python
   # Avant
   model = YOLO('yolov8n.pt')
   
   # Après
   model = YOLO('best.pt')
   ```
3. Dans `worker/Dockerfile`, ajoute avant CMD :
   ```dockerfile
   COPY best.pt .
   ```
4. Push sur GitHub → Railway redéploie automatiquement

✅ Ton worker utilise maintenant le modèle rugby custom !